**Welcome to the geochemistry biplot app for Bruker Results.csv files**

N. Tripcevich 2026, CC BY-SA 4.0  
[More Information Online](https://github.com/arf-berkeley/bruker-xrf-ppm-plot)


In [ ]:
%%capture
%pip install plotly ipywidgets
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.colors import DEFAULT_PLOTLY_COLORS
from IPython.display import display, HTML
import ipywidgets as widgets
import io

**Select the Results.csv table from your Bruker analysis**

Browse to a copy of the __Results.csv__ file typically found in Bruker/Data/Results.csv

Importing all of the Weight Percent data from the most recent Method used.

In [ ]:
from io import StringIO

# File upload
study_import = None
study = None

upload = widgets.FileUpload(accept='.csv', multiple=False)
confirm_btn = widgets.Button(
    description='Load Data',
    button_style='success',
    icon='check',
    disabled=True
)
status_label = widgets.Label('Please upload a Bruker XRF Results.csv file')

def parse_from_last_header(content_str):
    """Parse CSV from the last calibration header downwards.
    Bruker XRF instruments repeat the header row at the start of each
    calibration block - we only want data from the most recent one."""
    lines = content_str.splitlines(keepends=True)
    header_line = None
    for i in range(len(lines) - 1, -1, -1):
        if lines[i].split(',')[0].strip() == 'File #':
            header_line = i
            break
    if header_line is None:
        raise ValueError('Could not find header row in file')
    data = ''.join(lines[header_line:])
    df = pd.read_csv(StringIO(data))
    return df

def update_button(change):
    if upload.value:
        confirm_btn.disabled = False
        status_label.value = 'Results.csv ready - click Load Data to continue'
    else:
        confirm_btn.disabled = True
        status_label.value = 'Please upload a Bruker XRF Results.csv file'

def load_file(btn):
    global study_import
    try:
        content = upload.value[0]['content'].tobytes().decode('utf-8')
        study_import = parse_from_last_header(content)
        status_label.value = f'✓ Loaded {len(study_import)} rows - scroll down to filter'
        confirm_btn.disabled = True
        confirm_btn.description = 'Loaded'
        show_filters()
    except Exception as e:
        status_label.value = f'Error loading file: {e}'

upload.observe(update_button, names='value')
confirm_btn.on_click(load_file)

display(widgets.VBox([
    upload,
    confirm_btn,
    status_label
]))

Cleaning data includes removing the following: elemental error columns, Alloy, Match Qual columns, Multiplier, Cal Check, Operator, Field 1&2. This script also replaces Below Detection Limits LOD with 0.

You may now select rows of data from recent analyses by filtering with either File # or Date.

In [ ]:
def clean_data(df):
    # Metadata columns to keep
    META_COLS = ['File #', 'DateTime', 'Name', 'Application', 'Method', 'ElapsedTime']

    # Non-element columns to drop
    NON_ELEMENT_COLS = ['Alloy 1', 'Match Qual 1', 'Alloy 2', 'Match Qual 2', 
                        'Alloy 3', 'Match Qual 3', 'Multiplier', 'Cal Check',
                        'Operator', 'Field1', 'Field2']

    # Drop error columns and non-element columns
    drop_cols = [c for c in df.columns if 'Err' in c or c in NON_ELEMENT_COLS]
    keep_cols = [c for c in df.columns if c not in drop_cols]
    df = df[keep_cols].copy()

    # Replace below detection limit values with zero
    df = df.replace('< LOD', 0)

    # Ensure correct types
    string_cols = [c for c in ['Name', 'Application', 'Method'] if c in df.columns]
    numeric_cols = [c for c in df.columns if c not in string_cols + ['DateTime']]
    df[string_cols] = df[string_cols].astype('string')
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
    df['DateTime'] = pd.to_datetime(df['DateTime'])

    # Identify element columns (numeric cols excluding metadata)
    element_cols = [c for c in numeric_cols if c not in ['File #', 'ElapsedTime']]
    df[element_cols] = (df[element_cols] * 10000).round(1)

    # Drop rows and columns with all NaN
    df.dropna(axis=1, how='all', inplace=True)
    df.dropna(how='all', inplace=True)

    return df, element_cols

In [ ]:
filter_output = widgets.Output()
results_output = widgets.Output()

def show_filters():
    global study
    study = clean_data(study_import)[0]
    
    with filter_output:
        filter_output.clear_output(wait=True)
        
        min_file = int(study['File #'].min())
        max_file = int(study['File #'].max())
        min_date = study['DateTime'].min().date()
        max_date = study['DateTime'].max().date()

        file_slider = widgets.IntRangeSlider(
            value=[min_file, max_file],
            min=min_file,
            max=max_file,
            step=1,
            description='File #:',
            continuous_update=False,
            layout=widgets.Layout(width='500px')
        )

        start_picker = widgets.DatePicker(
            description='Start Date:',
            value=min_date,
            min=min_date,
            max=max_date
        )
        end_picker = widgets.DatePicker(
            description='End Date:',
            value=max_date,
            min=min_date,
            max=max_date
        )

        apply_btn = widgets.Button(
            description='Apply Filter',
            button_style='primary',
            icon='filter'
        )
        filter_status = widgets.Label(
            f'Data available: File # {min_file} to {max_file} | '
            f'{min_date} to {max_date}'
        )

        def apply_filter(btn):
            global study
            file_min, file_max = file_slider.value
            start = pd.Timestamp(start_picker.value)
            end = pd.Timestamp(end_picker.value) + pd.Timedelta(days=1)

            study = clean_data(study_import)[0]
            study = study[
                (study['File #'] >= file_min) &
                (study['File #'] <= file_max) &
                (study['DateTime'] >= start) &
                (study['DateTime'] < end)
            ].copy()

            filter_status.value = (
                f'✓ {len(study)} rows selected | '
                f'File # {file_min} to {file_max} | '
                f'{start.date()} to {end_picker.value}'
            )
            show_results()

        apply_btn.on_click(apply_filter)

        display(widgets.VBox([
            widgets.HTML('<b>Filter Data</b>'),
            file_slider,
            widgets.HBox([start_picker, end_picker]),
            apply_btn,
            filter_status
        ]))

display(filter_output)
display(results_output)

In [ ]:
def show_results():
    with results_output:
        results_output.clear_output(wait=True)

        if study is None or len(study) == 0:
            print('No data to display')
            return

        _, element_cols = clean_data(study_import)
        element_cols = [c for c in element_cols if c in study.columns]
        non_element = ['File #', 'DateTime', 'Name', 'ID']

        # --- Formatted Table ---
        display(widgets.HTML('<b>Results Table</b>'))
        display_df = study.copy()
        display_df['DateTime'] = display_df['DateTime'].dt.strftime('%m/%d/%Y %H:%M')
        text_cols = [c for c in ['DateTime', 'Name', 'ID'] if c in display_df.columns]
        display(display_df.style
            .format({col: '{:.1f}' for col in element_cols})
            .set_properties(**{'text-align': 'right', 'font-size': '12px'})
            .set_properties(subset=text_cols, **{'text-align': 'left'})
            .set_table_styles([{
                'selector': 'th',
                'props': [('text-align', 'center'), ('font-weight', 'bold')]
            }])
            .hide(axis='index')
        )

        # --- Biplot ---
        display(widgets.HTML('<br><b>Biplot</b>'))
        elements_present = [c for c in study.columns if c not in non_element]

        x_dropdown = widgets.Dropdown(
            options=elements_present,
            value='Sr' if 'Sr' in elements_present else elements_present[0],
            description='X Axis:'
        )
        y_dropdown = widgets.Dropdown(
            options=elements_present,
            value='Rb' if 'Rb' in elements_present else elements_present[1],
            description='Y Axis:'
        )
        plot_output = widgets.Output()

        def update_plot(change):
            with plot_output:
                plot_output.clear_output(wait=True)
                x = x_dropdown.value
                y = y_dropdown.value
                fig = px.scatter(
                    study,
                    x=x,
                    y=y,
                    color='Name',
                    hover_data=['File #', 'Name', 'DateTime', x, y],
                    title=f'{y} vs {x} Biplot',
                    labels={x: f'{x} (PPM)', y: f'{y} (PPM)'}
                )
                fig.update_traces(marker=dict(size=8))
                fig.update_layout(height=600, hovermode='closest')
                display(fig)

        x_dropdown.observe(update_plot, names='value')
        y_dropdown.observe(update_plot, names='value')

        display(widgets.VBox([
            widgets.HBox([x_dropdown, y_dropdown]),
            plot_output
        ]))
        update_plot(None)

        # --- Export ---
        display(widgets.HTML('<br><b>Export</b>'))
        export_output = widgets.Output()
        export_btn = widgets.Button(
            description='Export CSV',
            button_style='success',
            icon='download'
        )

        def on_export(btn):
            with export_output:
                export_output.clear_output(wait=True)
                csv_str = study.to_csv(index=False)
                b64 = __import__('base64').b64encode(csv_str.encode()).decode()
                filename = f'study_export_{study["DateTime"].max().strftime("%Y%m%d")}.csv'
                html = (
                    f'<a download="{filename}" '
                    f'href="data:text/csv;base64,{b64}">'
                    f'Click here to download {filename}</a>'
                )
                display(HTML(html))

        export_btn.on_click(on_export)
        display(widgets.VBox([export_btn, export_output]))